# Assignment 2 — Carlota Navarro

In [1]:
%pip install -q roboflow ultralytics

In [2]:
import os

try:
    from google.colab import userdata
    api_key = userdata.get("ROBOFLOW_API_KEY")
except ImportError:
    api_key = os.environ.get("ROBOFLOW_API_KEY")

if not api_key:
    raise RuntimeError(
        "No ROBOFLOW_API_KEY found. In Colab: add it under Secrets (key icon). "
        "Elsewhere: export ROBOFLOW_API_KEY=... or put it in .env."
    )
print("API key loaded:", api_key[:4] + "...")

API key loaded: 9aw5...


In [3]:
from roboflow import Roboflow

WORKSPACE = "roboflow-universe-projects"
PROJECT = "banana-ripeness-classification"
VERSION = 1

rf = Roboflow(api_key=api_key)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("folder")
print("Downloaded to:", dataset.location)

loading Roboflow workspace...
loading Roboflow project...
Downloaded to: /content/Banana-Ripeness-Classification-1


In [4]:
import os

for split in ["train", "valid", "test"]:
    split_dir = os.path.join(dataset.location, split)
    if not os.path.isdir(split_dir):
        continue
    print(f"\n{split.upper()}")
    classes = sorted(os.listdir(split_dir))
    for c in classes:
        class_dir = os.path.join(split_dir, c)
        if os.path.isdir(class_dir):
            n = len([f for f in os.listdir(class_dir) if not f.startswith(".")])
            print(f"  {c}: {n} images")


TRAIN
  freshripe: 704 images
  freshunripe: 501 images
  overripe: 783 images
  ripe: 470 images
  rotten: 1340 images
  unripe: 133 images

VALID
  freshripe: 207 images
  freshunripe: 136 images
  overripe: 229 images
  ripe: 132 images
  rotten: 388 images
  unripe: 31 images

TEST
  freshripe: 102 images
  freshunripe: 83 images
  overripe: 113 images
  ripe: 52 images
  rotten: 185 images
  unripe: 27 images


In [5]:
#count classes after merging, for the dataset description
import os

RAW_ROOT = dataset.location
MERGE_MAP = {
    "unripe": ["unripe", "freshunripe"],
    "ripe": ["ripe", "freshripe"],
    "rotten": ["rotten", "overripe"],
}

counts = {}
for split in ["train", "valid", "test"]:
    split_dir = os.path.join(RAW_ROOT, split)
    counts[split] = {}
    for merged_class, raw_classes in MERGE_MAP.items():
        total = 0
        for rc in raw_classes:
            class_dir = os.path.join(split_dir, rc)
            if os.path.isdir(class_dir):
                total += len([f for f in os.listdir(class_dir) if not f.startswith(".")])
        counts[split][merged_class] = total

for split, class_counts in counts.items():
    split_total = sum(class_counts.values())
    print(f"\n{split.upper()} — {split_total} images total")
    for c, n in class_counts.items():
        print(f"  {c}: {n} ({100*n/split_total:.1f}%)")


TRAIN — 3931 images total
  unripe: 634 (16.1%)
  ripe: 1174 (29.9%)
  rotten: 2123 (54.0%)

VALID — 1123 images total
  unripe: 167 (14.9%)
  ripe: 339 (30.2%)
  rotten: 617 (54.9%)

TEST — 562 images total
  unripe: 110 (19.6%)
  ripe: 154 (27.4%)
  rotten: 298 (53.0%)


In [6]:
#see how many images of each type there are in the splits
overall_total = sum(sum(c.values()) for c in counts.values())
print(f"Overall total: {overall_total} images")
for split, class_counts in counts.items():
    st = sum(class_counts.values())
    print(f"  {split}: {st} ({100*st/overall_total:.1f}% of total)")

#see imbalance in the train data
train_counts = counts["train"]
max_c = max(train_counts, key=train_counts.get)
min_c = min(train_counts, key=train_counts.get)
ratio = train_counts[max_c] / train_counts[min_c]
print(f"\nTrain-set imbalance: {max_c} outnumbers {min_c} by {ratio:.1f}x "
      f"({train_counts[max_c]} vs {train_counts[min_c]})")

Overall total: 5616 images
  train: 3931 (70.0% of total)
  valid: 1123 (20.0% of total)
  test: 562 (10.0% of total)

Train-set imbalance: rotten outnumbers unripe by 3.3x (2123 vs 634)


In [7]:
from PIL import Image
import random

sizes = set()
for split in ["train", "valid", "test"]:
    split_dir = os.path.join(RAW_ROOT, split)
    for c in os.listdir(split_dir):
        class_dir = os.path.join(split_dir, c)
        files = os.listdir(class_dir)
        for f in random.sample(files, min(5, len(files))):
            with Image.open(os.path.join(class_dir, f)) as img:
                sizes.add(img.size)
print("Observed image sizes (sample across classes/splits):", sizes)

Observed image sizes (sample across classes/splits): {(480, 640), (480, 360), (640, 480)}


In [8]:
import os, shutil

MERGE_MAP = {
    "unripe": ["unripe", "freshunripe"],
    "ripe": ["ripe", "freshripe"],
    "rotten": ["rotten", "overripe"],
}
SPLIT_MAP = {"train": "train", "valid": "val", "test": "test"}

MERGED_ROOT = os.path.join(os.path.dirname(dataset.location), "banana_merged")
if os.path.exists(MERGED_ROOT):
    shutil.rmtree(MERGED_ROOT)

for src_split, dst_split in SPLIT_MAP.items():
    src_split_dir = os.path.join(dataset.location, src_split)
    if not os.path.isdir(src_split_dir):
        continue
    for merged_class, source_classes in MERGE_MAP.items():
        dst_dir = os.path.join(MERGED_ROOT, dst_split, merged_class)
        os.makedirs(dst_dir, exist_ok=True)
        for sc in source_classes:
            src_dir = os.path.join(src_split_dir, sc)
            if not os.path.isdir(src_dir):
                continue
            for fname in os.listdir(src_dir):
                if fname.startswith("."):
                    continue
                shutil.copy2(os.path.join(src_dir, fname), os.path.join(dst_dir, f"{sc}_{fname}"))

print("Merged dataset built at:", MERGED_ROOT)

Merged dataset built at: /content/banana_merged


In [9]:
for split in ["train", "val", "test"]:
    split_dir = os.path.join(MERGED_ROOT, split)
    print(f"\n{split.upper()}")
    for c in sorted(os.listdir(split_dir)):
        print(f"  {c}: {len(os.listdir(os.path.join(split_dir, c)))}")


TRAIN
  ripe: 1174
  rotten: 2123
  unripe: 634

VAL
  ripe: 339
  rotten: 617
  unripe: 167

TEST
  ripe: 154
  rotten: 298
  unripe: 110


In [10]:
from ultralytics import YOLO

model = YOLO("yolo11n-cls.pt")
results = model.train(
    data=MERGED_ROOT,
    epochs=30,
    imgsz=224,
    patience=10,
    project="runs_cls",
    name="banana_ripeness",
    seed=0,
)

Ultralytics 8.4.184 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/banana_merged, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=banana_ripeness, nbs=64, nms=None, opset=None, optimize=False, 

In [11]:
best_path = str(results.save_dir / "weights" / "best.pt")
print("Best weights at:", best_path)

os.makedirs("weights", exist_ok=True)
shutil.copy2(best_path, "weights/banana_ripeness_cls.pt")
print("Saved to weights/banana_ripeness_cls.pt")

from google.colab import files
files.download("weights/banana_ripeness_cls.pt")


Best weights at: /content/runs/classify/runs_cls/banana_ripeness/weights/best.pt
Saved to weights/banana_ripeness_cls.pt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
test_dir = os.path.join(MERGED_ROOT, "test")
test_items = []  # (filepath, true_label)
for c in sorted(os.listdir(test_dir)):
    for fname in os.listdir(os.path.join(test_dir, c)):
        test_items.append((os.path.join(test_dir, c, fname), c))
print(f"{len(test_items)} test images")

562 test images


In [13]:
#trivial baseline - always predict majority training class

train_dir = os.path.join(MERGED_ROOT, "train")
class_counts = {c: len(os.listdir(os.path.join(train_dir, c))) for c in os.listdir(train_dir)}
majority_class = max(class_counts, key=class_counts.get)
print("Majority class:", majority_class, class_counts)

trivial_acc = sum(true_label == majority_class for _, true_label in test_items) / len(test_items)
print(f"Trivial baseline accuracy: {trivial_acc:.3f}")

Majority class: rotten {'unripe': 634, 'rotten': 2123, 'ripe': 1174}
Trivial baseline accuracy: 0.530


In [14]:
#Zero-shot pretrained baseline

import pandas as pd
from ultralytics import YOLO

zeroshot_model = YOLO("yolo11n-cls.pt")  #untouched ImageNet weights, no finetuning
zs_names = zeroshot_model.names

zs_top1 = []
for fpath, true_label in test_items:
    r = zeroshot_model.predict(fpath, verbose=False)[0]
    zs_top1.append(zs_names[r.probs.top1])

true_labels = [t for _, t in test_items]
df = pd.DataFrame({"true_label": true_labels, "zs_prediction": zs_top1})

#Overall how often does it even recognize the images as a banana?
banana_rate = (df["zs_prediction"] == "banana").mean()
print(f"Recognized as 'banana' overall: {banana_rate:.1%} ({(df['zs_prediction']=='banana').sum()}/{len(df)})")

#Broken down by true ripeness class
print("\nRecognition rate by true ripeness class:")
print(df.groupby("true_label")["zs_prediction"].apply(lambda x: (x == "banana").mean()).round(3))

#For the ones it did not call 'banana', what did it guess instead, per class?
print("\nWhat it guessed instead, for images it failed to recognize as 'banana':")
misses = df[df["zs_prediction"] != "banana"]
for label in sorted(misses["true_label"].unique()):
    print(f"\n  True label = {label}:")
    print(misses[misses["true_label"] == label]["zs_prediction"].value_counts().head(5).to_string())

Recognized as 'banana' overall: 65.1% (366/562)

Recognition rate by true ripeness class:
true_label
ripe      0.974
rotten    0.634
unripe    0.245
Name: zs_prediction, dtype: float64

What it guessed instead, for images it failed to recognize as 'banana':

  True label = ripe:
zs_prediction
bolete              2
spaghetti_squash    1
paddle              1

  True label = rotten:
zs_prediction
nematode       19
slug           18
eel            13
cucumber       11
common_newt     6

  True label = unripe:
zs_prediction
cucumber        64
zucchini        15
Granny_Smith     3
ocarina          1


In [15]:
#My model’s real test accuracy and per-class breakdown:

import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

best_model = YOLO("weights/banana_ripeness_cls.pt")
class_names = best_model.names
rotten_idx = [i for i, n in class_names.items() if n == "rotten"][0]

y_true, y_pred, probs_rotten = [], [], []
for fpath, true_label in test_items:
    r = best_model.predict(fpath, verbose=False)[0]
    y_true.append(true_label)
    y_pred.append(class_names[r.probs.top1])
    probs_rotten.append(float(r.probs.data[rotten_idx]))

test_acc = sum(p == t for p, t in zip(y_pred, y_true)) / len(y_true)
print(f"Finetuned model test accuracy: {test_acc:.3f}")
print(classification_report(y_true, y_pred, digits=3))
labels_sorted = sorted(set(y_true))
print("Confusion matrix (rows=true, cols=pred), order:", labels_sorted)
print(confusion_matrix(y_true, y_pred, labels=labels_sorted))

Finetuned model test accuracy: 0.991
              precision    recall  f1-score   support

        ripe      1.000     0.994     0.997       154
      rotten      0.997     0.987     0.992       298
      unripe      0.965     1.000     0.982       110

    accuracy                          0.991       562
   macro avg      0.987     0.993     0.990       562
weighted avg      0.991     0.991     0.991       562

Confusion matrix (rows=true, cols=pred), order: ['ripe', 'rotten', 'unripe']
[[153   1   0]
 [  0 294   4]
 [  0   0 110]]


In [16]:
import csv

os.makedirs("artifacts", exist_ok=True)
with open("artifacts/test_predictions.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["filepath", "true_label", "pred_label", "prob_rotten"])
    for (fpath, true_label), pred_label, p_rotten in zip(test_items, y_pred, probs_rotten):
        writer.writerow([fpath, true_label, pred_label, p_rotten])
print("Saved artifacts/test_predictions.csv —", len(test_items), "rows")

from google.colab import files
files.download("artifacts/test_predictions.csv")

Saved artifacts/test_predictions.csv — 562 rows


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>